In [ ]:
import pandas as pd
import numpy as np
import sys
import os

# Add the src directory to Python path
# From fig_4_ab/ go up 5 levels to reach src/
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', '..', '..', '..'))
src_path = os.path.join(project_root, 'src')
sys.path.insert(0, src_path)

from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from mimiciii_db import DB
from mimiciii_db.config import db_url


In [ ]:
db = DB.from_url(db_url())
print("Database connected successfully!")


In [ ]:
lca_path = os.getenv("LCA_PATH")
lca = pd.read_csv(lca_path)


In [ ]:
filtered_patients_with_morbidity_counts_path = os.getenv("FILTERED_PATIENTS_WITH_MORBIDITY_COUNTS_PATH")
filtered_patients_with_morbidity_counts = db.table_df(filtered_patients_with_morbidity_counts_path, schema="mimiciii")

In [ ]:
# --- Load SOFA and OASIS data
sofa_path = os.getenv("SOFA_PATH")
oasis_path = os.getenv("OASIS_PATH")
sofa_df  = pd.read_csv(sofa_path)
oasis_df = pd.read_csv(oasis_path)

patients = filtered_patients_with_morbidity_counts.copy()

# --- Ensure key dtypes match
keys = ['subject_id','hadm_id']
for df in (patients, sofa_df, oasis_df):
    for k in keys:
        df[k] = pd.to_numeric(df[k], errors='coerce')

# keep first; or replace with .groupby(keys).agg({'sofa_total':'max', ...})
sofa_df  = sofa_df.drop_duplicates(subset=keys)
oasis_df = oasis_df.drop_duplicates(subset=keys)

# --- Left join SOFA then OASIS
merged = (
    patients
      .merge(sofa_df,  on=keys, how='left', suffixes=('', '_sofa'))
      .merge(oasis_df, on=keys, how='left', suffixes=('', '_oasis'))
)

print("patients rows:", len(patients))
print("merged rows  :", len(merged))        # should be the same
print("new columns  :", [c for c in merged.columns if c not in patients.columns])


In [ ]:
df_merged = merged.merge(lca[['hadm_id','latent_class']],
                            on='hadm_id', how='left')


In [ ]:
import matplotlib.pyplot as plt

order  = [1,2,3,4,5,6]
colors = ['#FFFFFF', '#E41A1C', '#4DAF4A', '#377EB8', '#4DD2D2', '#E377C2']  # 1–6

def colored_boxplot(ax, ycol):
    data = [df_merged.loc[df_merged['latent_class']==k, ycol].dropna() for k in order]
    bp = ax.boxplot(data, labels=order, showfliers=True, patch_artist=True)
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c)
        patch.set_edgecolor('black')
    for w in bp['whiskers'] + bp['caps']:
        w.set_color('black')
    for m in bp['medians']:
        m.set_color('black'); m.set_linewidth(2)
    ax.set_xlabel('Subgroup')

fig, axes = plt.subplots(1, 2, figsize=(10,4))
colored_boxplot(axes[0], 'sofa');  axes[0].set_title('A'); axes[0].set_ylabel('SOFA score')
colored_boxplot(axes[1], 'oasis'); axes[1].set_title('B'); axes[1].set_ylabel('OASIS score')
plt.tight_layout()

# Save figure
import os
plt.savefig("../assets/fig_4/fig_4ab.png")
plt.show()
